In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True

DB_PATH = "transcriber.db"

conn = sqlite3.connect(DB_PATH)

videos = pd.read_sql_query("SELECT * FROM videos", conn)
submissions = pd.read_sql_query("SELECT * FROM submissions", conn)
analyses = pd.read_sql_query("SELECT * FROM analyses", conn)

conn.close()

# Parse timestamps
for df in (videos, submissions, analyses):
    for col in ["first_seen_at", "submitted_at", "created_at", "opened_at", "deleted_at"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce", utc=True)

print("Videos:", len(videos))
print("Submissions:", len(submissions))
print("Analyses:", len(analyses))

In [ ]:
is_followup = analyses["parent_id"].notna()

followup_rate = is_followup.mean()

print(f"Follow-up analyses: {is_followup.sum()}")
print(f"Initial analyses: {(~is_followup).sum()}")
print(f"Follow-up rate: {followup_rate:.1%}")

In [ ]:
local = submissions.copy()

local["local_time"] = (
    local["submitted_at"]
    .dt.tz_convert("Europe/Moscow")
)

local["hour"] = local["local_time"].dt.hour

hourly = local.groupby("hour").size()

ax = hourly.plot(
    kind="bar",
    figsize=(12, 5)
)

ax.set_title("When do I use the transcriber?")
ax.set_xlabel("Hour of day")
ax.set_ylabel("Submissions")

plt.show()

local["weekday"] = local["local_time"].dt.day_name()

weekday_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

heatmap = (
    local
    .pivot_table(
        index="weekday",
        columns="hour",
        values="id",
        aggfunc="count",
        fill_value=0
    )
    .reindex(weekday_order)
)

plt.figure(figsize=(14, 5))

plt.imshow(heatmap, aspect="auto")

plt.colorbar(label="Submissions")

plt.xticks(range(len(heatmap.columns)), heatmap.columns)
plt.yticks(range(len(heatmap.index)), heatmap.index)

plt.xlabel("Hour")
plt.ylabel("Day")
plt.title("When I use the transcriber")

plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

submissions["coverage"].dropna().plot(
    kind="hist",
    bins=30,
    ax=axes[0]
)
axes[0].set_title("Caption coverage")
axes[0].set_xlabel("Coverage")

submissions["density"].dropna().plot(
    kind="hist",
    bins=30,
    ax=axes[1]
)
axes[1].set_title("Caption density")
axes[1].set_xlabel("Density")

submissions["words_per_min"].dropna().plot(
    kind="hist",
    bins=30,
    ax=axes[2]
)
axes[2].set_title("Caption words/min")
axes[2].set_xlabel("Words/min")

plt.tight_layout()
plt.show()

In [ ]:
timing_cols = [
    "t_metadata",
    "t_captions",
    "t_download",
    "t_whisper",
]

timings = submissions[timing_cols].mean().sort_values()

ax = timings.plot(
    kind="bar",
    figsize=(10, 5)
)

ax.set_title("Average time spent in each transcription stage")
ax.set_ylabel("Seconds")
ax.tick_params(axis="x", rotation=30)

plt.show()

In [ ]:
merged = submissions.merge(
    videos[["video_id", "duration"]],
    on="video_id",
    how="left"
)

plt.figure(figsize=(10, 6))

plt.scatter(
    merged["duration"] / 60,
    merged["t_total"],
    alpha=0.5
)

plt.xlabel("Video duration (minutes)")
plt.ylabel("Processing time (seconds)")
plt.title("Video duration vs transcription time (not whisper)")

plt.show()

whisper = merged[
    merged["source"].fillna("").str.startswith("whisper")
].dropna(subset=["duration", "t_whisper"])

plt.figure(figsize=(10, 6))

plt.scatter(
    whisper["duration"] / 60,
    whisper["t_whisper"],
    alpha=0.5
)

plt.xlabel("Video duration (minutes)")
plt.ylabel("Whisper time (seconds)")
plt.title("Video duration vs Whisper processing time")

plt.show()

In [ ]:
workflow = analyses.merge(
    submissions[
        ["video_id", "submitted_at"]
    ],
    on="video_id",
    how="left"
)

workflow["analysis_delay_hours"] = (
    workflow["created_at"] -
    workflow["submitted_at"]
).dt.total_seconds() / 3600

workflow["read_delay_hours"] = (
    workflow["opened_at"] -
    workflow["created_at"]
).dt.total_seconds() / 3600

print(
    workflow[
        ["analysis_delay_hours", "read_delay_hours"]
    ].describe()
)

plt.figure(figsize=(10, 5))

plt.hist(
    workflow["read_delay_hours"].dropna(),
    bins=40
)

plt.xlabel("Hours between analysis creation and opening")
plt.ylabel("Analyses")
plt.title("How long do analyses sit before I read them?")

plt.show()